# 协同设计模式


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 分工原则：按数据形态切段

一个算子的计算图可以逐段问三个问题，答案决定该段归谁：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">问题</th><th style="text-align:left;">偏 SIMD</th><th style="text-align:left;">偏 SIMT</th></tr>
<tr><td>数据访问规整吗？</td><td>连续、同构、可分块</td><td>离散索引、逐元素变址</td></tr>
<tr><td>操作一致吗？</td><td>同一操作作用于整块</td><td>逐元素分支/条件不同</td></tr>
<tr><td>输出位置确定吗？</td><td>规则（out[i,j]）</td><td>运行时决定（scatter/竞争）</td></tr>
</table>

**稠密段 SIMD、稀疏段 SIMT、两者由外层 Kernel 拼接**——分工的单位是"计算段"，不是整个算子。


## 2. 组织骨架：外层 Kernel 是"总装车间"

回顾第 6 章结论：本课程采用 SIMT VF 嵌于外层 Kernel 的混合路径。协同骨架因此天然成立：

```text
外层 @pl.jit Kernel（一个总装车间）
├── Tile/TileGroup 规划（SIMD 世界：分块、双缓冲、地址）
├── section_vector（SIMT 只在 Vector 域调用）；section_cube（可选）
│     ├── SIMD 段：load / pl.add / store …（Vector Tile 批量）；matmul 放在可选的 section_cube 中
│     ├── SIMT 段：simt_func[threads](...)（逐线程，tile[row,col] 元素级）
│     └── 两段之间的数据交接（UB Tile 或 GM Tensor）
└── 同步：auto_mutex（核内）+ 事件（跨核，若有 Cube 参与）
```

三种段间交接方式：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">交接</th><th style="text-align:left;">载体</th><th style="text-align:left;">适用</th></tr>
<tr><td>SIMD 段 → SIMT 段</td><td>UB Tile（SIMT 以 tile[row,col] 访问）</td><td>稠密结果供逐线程筛选/变址</td></tr>
<tr><td>SIMT 段 → SIMD 段</td><td>UB Tile / GM Tensor</td><td>筛选/索引结果供批量计算</td></tr>
<tr><td>SIMT 段 → SIMT 段</td><td>共享 Tile + 原子</td><td>块内协作（第 6 章初始化、两次屏障、原子更新及读取模式）</td></tr>
</table>


## 3. 一个最小协同示例：筛选后缩放

`y[i] = x[i] * 2.0（仅当 x[i] > 0，其余置 0）`——可以两种写法对照：

```python
# 纯 SIMD：掩码选择（第 3 章模式）
pl.gt(mask_tile, x_tile, zero_tile)
pl.mul(tmp_tile, x_tile, two_tile)
pl.select(out_tile, mask_tile, tmp_tile, zero_tile, tmp2_tile)

# 协同：SIMD 搬入 → SIMT 逐元素判断 →（可继续 SIMD 搬出）
@pl.vector_function(mode="simt", max_threads=256)
def filter_scale(x_tile, out_tile, count: pl.DT_UINT32):
    # UB 是当前核私有 Tile：用块内索引，不能加全局 block 偏移
    tid = pl.simt.linear_thread_idx()
    for i in pl.range(tid, count, 256):
        v = x_tile[0, i]
        if v > 0.0:
            out_tile[0, i] = v * 2.0
        else:
            out_tile[0, i] = 0.0
```

**调用前提**：按 [256] 启动，count 不超过 x/out Tile 的有效列数；外层按核处理互不相交的数据。以上是片段，需补常量 Tile 和搬运。

**教学点**：两版均实现筛选后乘 2，SIMD 掩码更简洁——**协同不是目的，分工匹配才是**。当"筛选条件"复杂到掩码难表达（如查表、多级条件、索引计算）时，协同才值得进一步评估。


## 4. 设计流程（五步）

1. **画计算图**：把算子拆成段，标注每段的数据形态（规整/离散）；
2. **段分类**：三个问题定归属（SIMD/SIMT/混合段内部再拆）；
3. **定交接载体**：UB Tile（快、口径一致）或 GM workspace（大、灵活）；
4. **组织外层 Kernel**：Tile 规划、段调用顺序、同步（auto_mutex + 必要时事件）；
5. **性能预算**：估算各段耗时与交接开销，决定是否值得流水化（第 5 章/第 8 章）。


## 5. 完整推演：8 专家门控 FFN 的数据流

设定：batch=4、隐藏维 D=512、专家数 E=8、每样本选 top-2 专家。逐步推演各段的数据规模与接口选择：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">步骤</th><th style="text-align:left;">数据</th><th style="text-align:left;">形状/规模</th><th style="text-align:left;">归属</th><th style="text-align:left;">接口要点</th></tr>
<tr><td>① 门控打分</td><td>x·W_g</td><td>[4, 512]×[512, 8] → [4, 8]</td><td>Cube（小 matmul）</td><td>逻辑上 8 列对应 8 专家；需处理分形对齐，K=512 通常分块，不能直接假定单 Tile 放入 L0</td></tr>
<tr><td>② 逐样本 top-2</td><td>g</td><td>[4, 8] → 每行取 top-2</td><td>SIMT</td><td>4 线程，每线程比较 8 个分数（线程数少但逻辑逐个——正好是 SIMT 特征）</td></tr>
<tr><td>③ 竞争取号</td><td>选中标记</td><td>总共 8 条路由记录；每个专家桶最多 4 项</td><td>SIMT</td><td>atomic_add 取紧凑位置（第 9 章实践一模式）</td></tr>
<tr><td>④ 紧凑化路由</td><td>x 行按专家分组</td><td>[4, 512] → [≤8, 512]（每专家子块）</td><td>SIMT（Gather 行）</td><td>每线程一行 scatter 到目标专家段</td></tr>
<tr><td>⑤ 分组 GEMM</td><td>各专家段 × W_e</td><td>8 段小 GEMM</td><td>Cube</td><td>段内规整 → 逐段 matmul（段边界 = 元数据）</td></tr>
<tr><td>⑥ 汇聚输出</td><td>各段结果回原位</td><td>[≤8, 512] → [4, 512]</td><td>SIMT（scatter 回）</td><td>记录原行号及门控权重；top-2 两路需加权求和，不能直接竞争覆写同一输出行</td></tr>
</table>

**推演收获**：步骤②③⑥的"逐个判断/竞争/按元数据回写"正是第 7 章三大场景的直接复用；步骤①⑤是第 4 章 matmul——协同算子的设计就是**把计算图拆成段，每段对号入座**。


### 完整 UB 交接主例

输入 [1,1024] FP32，用一个 Vector 核：SIMD load 将数据搬入 4 KiB 的 UB Tile，256 个 SIMT 线程各处理 4 个元素，再由 SIMD store 写回。线程对当前核私有 Tile 使用块内索引；数据所在的 UB TileGroup 让外层自动同步能够识别搬运与 SIMT 调用的依赖。

下面示例与 `relu(x)*2` 等价，用它先验证机制。若条件始终如此简单，纯 Tile 实现也值得比较；引入 SIMT 的理由应来自实际的不规则计算，而非形式上的混用。


In [ ]:
@pl.vector_function(mode="simt",max_threads=256)
def filter_local(x,y):
    tid=pl.simt.linear_thread_idx()
    for i in pl.range(tid,1024,256):
        value=x[0,i]
        if value > 0.0:
            y[0,i]=value*2.0
        else:
            y[0,i]=0.0


In [ ]:
@pl.jit(auto_mutex=True)
def hybrid_scale(x: pl.Tensor[[1,1024],pl.DT_FP32],y: pl.Tensor[[1,1024],pl.DT_FP32]):
    tt=pl.TileType(shape=[1,1024],dtype=pl.DT_FP32,target_memory=pl.MemorySpace.Vec)
    gx=pl.make_tile_group(type=tt,addrs=0,mutex_ids=[0])
    gy=pl.make_tile_group(type=tt,addrs=0x1000,mutex_ids=[1])
    with pl.section_vector():
        tx=gx.current()
        ty=gy.current()
        pl.load(tx,x,[0,0])
        filter_local[256](tx,ty)
        pl.store(y,ty,[0,0])


## 6. 课后练习

1. （填空）分工三问：数据访问 ______、操作 ______、输出位置 ______。
2. （选择）"每行数据的列数存在一张变长表里，需逐行按表取数"——这段计算归：
   A. SIMD<br/>B. SIMT<br/>C. Cube
3. （实践）把 RMSNorm（第 3 章）按五步流程走一遍：哪些段天然 SIMD？有没有必要引入 SIMT？给出结论与理由。

In [ ]:
!cat ./answer/09.02_answer.txt

## 7. 本节小结

- 分工三问（访问/操作/输出）按**计算段**分配 SIMD/SIMT；
- 外层 Kernel 总装：Tile 规划 + 段调用 + 交接（UB Tile 为主）+ 同步；
- 协同是手段；掩码能表达时先比较 Tile 与 SIMT 的实现成本及实测收益；
- 五步设计流程：画图 → 分类 → 定交接 → 组织 → 预算。